In [19]:
import pandas as pd

In [20]:
df=pd.read_csv("hotel_bookings.csv")

In [31]:
#Task 13 – Hotel Booking Demand Understanding, Cleaning & Reservation Analytics
#Clean missing values in categorical columns such as country, agent, and company. Calculate total length of stay and
#total guest headcounts by combining adults, children, and babies. Compute cancellation percentages across City Hotel and Resort Hotel, and 
#detect outliers in lead times and Average Daily Rate (ADR).


In [ ]:
df

In [ ]:
df.shape

In [22]:
df.head()

,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,...,No Deposit,NaN,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
3,Resort Hotel,0,13,2015,July,27,1,0,1,1,...,No Deposit,304.0,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,...,No Deposit,240.0,NaN,0,Transient,98.0,0,1,Check-Out,2015-07-03


In [23]:
df.tail()

,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
119385,City Hotel,0,23,2017,August,35,30,2,5,2,...,No Deposit,394.0,NaN,0,Transient,96.14,0,0,Check-Out,2017-09-06
119386,City Hotel,0,102,2017,August,35,31,2,5,3,...,No Deposit,9.0,NaN,0,Transient,225.43,0,2,Check-Out,2017-09-07
119387,City Hotel,0,34,2017,August,35,31,2,5,2,...,No Deposit,9.0,NaN,0,Transient,157.71,0,4,Check-Out,2017-09-07
119388,City Hotel,0,109,2017,August,35,31,2,5,2,...,No Deposit,89.0,NaN,0,Transient,104.40,0,0,Check-Out,2017-09-07
119389,City Hotel,0,205,2017,August,35,29,2,7,2,...,No Deposit,9.0,NaN,0,Transient,151.20,0,2,Check-Out,2017-09-07


In [24]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 119390 entries, 0 to 119389
Data columns (total 32 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   hotel                           119390 non-null  object 
 1   is_canceled                     119390 non-null  int64  
 2   lead_time                       119390 non-null  int64  
 3   arrival_date_year               119390 non-null  int64  
 4   arrival_date_month              119390 non-null  object 
 5   arrival_date_week_number        119390 non-null  int64  
 6   arrival_date_day_of_month       119390 non-null  int64  
 7   stays_in_weekend_nights         119390 non-null  int64  
 8   stays_in_week_nights            119390 non-null  int64  
 9   adults                          119390 non-null  int64  
 10  children                        119386 non-null  float64
 11  babies                          119390 non-null  int64  
 12  meal            

In [25]:
df.isnull().sum()

hotel                                  0
is_canceled                            0
lead_time                              0
arrival_date_year                      0
arrival_date_month                     0
arrival_date_week_number               0
arrival_date_day_of_month              0
stays_in_weekend_nights                0
stays_in_week_nights                   0
adults                                 0
children                               4
babies                                 0
meal                                   0
country                              488
market_segment                         0
distribution_channel                   0
is_repeated_guest                      0
previous_cancellations                 0
previous_bookings_not_canceled         0
reserved_room_type                     0
assigned_room_type                     0
booking_changes                        0
deposit_type                           0
agent                              16340
company         

In [26]:
df["country"]=df["country"].fillna(df["country"].mode()[0])

In [27]:
df["agent"]=df["agent"].fillna("unknown")                          

In [28]:
df["company"]=df["company"].fillna("unknown") 

In [33]:
df[["country","agent","company"]].isnull().sum()

country    0
agent      0
company    0
dtype: int64

In [34]:
#total legth of stay
df["total_stay"]=(df["stays_in_weekend_nights"]+df["stays_in_week_nights"])

In [36]:
df["total_stay"]

0         0
1         0
2         1
3         1
4         2
         ..
119385    7
119386    7
119387    7
119388    7
119389    9
Name: total_stay, Length: 119390, dtype: int64

In [39]:
#replace missing values
df["children"]=df["children"].fillna(0)

In [42]:
#total guest 
df["total_guest"]=(df["adults"] + df["children"] + df["babies"])

In [44]:
print(df[["total_guest","total_stay"]])

        total_guest  total_stay
0               2.0           0
1               2.0           0
2               1.0           1
3               1.0           1
4               2.0           2
...             ...         ...
119385          2.0           7
119386          3.0           7
119387          2.0           7
119388          2.0           7
119389          2.0           9

[119390 rows x 2 columns]


In [47]:
cancellation=df.groupby("hotel")["is_canceled"].agg(
    ["sum","count"])
cancellation["cancellation_percentage"]=(cancellation["sum"]/cancellation["count"])*100
print(cancellation)

                sum  count  cancellation_percentage
hotel                                              
City Hotel    33102  79330                41.726963
Resort Hotel  11122  40060                27.763355


In [48]:
def detect_outliers(column):
    Q1=df[column].quantile(0.25)
    Q3=df[column].quantile(0.75)
    IQR=Q3-Q1

    lower_limit=Q1-1.5*IQR
    upper_limit=Q3+1.5*IQR

    outliers=df[
        (df[column]<lower_limit) |
        (df[column]>lower_limit) ]
    print("column:",column)
    print("Q1:",Q1)
    print("Q3:",Q3)
    print("lower limit:",lower_limit)
    print("upper_limit",upper_limit)
    print("number of outliers:",len(outliers))
    print()

    return outliers
    

In [49]:
lead_time_outliers= detect_outliers("lead_time")

column: lead_time
Q1: 18.0
Q3: 160.0
lower limit: -195.0
upper_limit 373.0
number of outliers: 119390



In [50]:
lead_time_outliers= detect_outliers("adr")

column: adr
Q1: 69.29
Q3: 126.0
lower limit: -15.774999999999991
upper_limit 211.065
number of outliers: 119390

